In [ ]:
#!/usr/bin/env python3
import numpy as np

from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_algorithms.optimizers import SLSQP
from qiskit_algorithms.minimum_eigensolvers import VQE

print(" FULL H₂ VQE — Electronic + Nuclear Repulsion (with custom correlated noise)")


# =====================================================
#  YOUR CORRELATED NOISE MODELS (TOPAZ-STYLE)
# =====================================================
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1],
              [1, 0]], dtype=complex)
Y = np.array([[0, -1j],
              [1j, 0]], dtype=complex)
Z = np.array([[1, 0],
              [0, -1]], dtype=complex)


def build_full_operator_from_list(oplist):
    if not oplist:
        return np.eye(1, dtype=complex)
    fullop = oplist[0]
    for qidx in range(1, len(oplist)):
        fullop = np.kron(fullop, oplist[qidx])
    return fullop


def apply_correlated_depolarizing_noise(rho_ideal,
                                        n_qubits,
                                        base_strength,
                                        correlation_length,
                                        correlation_strength):
    rho_current = rho_ideal.copy()

    # Single-qubit depolarizing
    for qubit in range(n_qubits):
        p_single = base_strength
        K_ops = [
            np.sqrt(1 - 3 * p_single / 4) * I2,
            np.sqrt(p_single / 4) * X,
            np.sqrt(p_single / 4) * Y,
            np.sqrt(p_single / 4) * Z,
        ]
        rho_qubit_noisy = np.zeros_like(rho_current, dtype=complex)
        for Kq in K_ops:
            oplist = [I2] * n_qubits
            oplist[qubit] = Kq
            K_full = build_full_operator_from_list(oplist)
            rho_qubit_noisy += K_full @ rho_current @ K_full.conj().T
        rho_current = rho_qubit_noisy

    # Correlated two-qubit depolarizing (nearest neighbours)
    for qubit1 in range(n_qubits - 1):
        qubit2 = qubit1 + 1
        distance = 1.0
        p_corr = correlation_strength * base_strength * np.exp(-distance / correlation_length)
        if p_corr < 1e-12:
            continue

        rho_2q_noisy = (1.0 - p_corr) * rho_current
        for P1, P2 in [(X, X), (Y, Y), (Z, Z)]:
            oplist = [I2] * n_qubits
            oplist[qubit1] = P1
            oplist[qubit2] = P2
            P_full = build_full_operator_from_list(oplist)
            rho_2q_noisy += (p_corr / 3.0) * (P_full @ rho_current @ P_full.conj().T)
        rho_current = rho_2q_noisy

    tr = np.trace(rho_current)
    if abs(tr) > 1e-12:
        rho_current = rho_current / tr
    return rho_current


def apply_correlated_amplitude_damping(rho_ideal,
                                       n_qubits,
                                       gamma_base,
                                       correlation_length,
                                       correlation_strength):
    rho_current = rho_ideal.copy()

    K0 = np.array([[1, 0],
                   [0, np.sqrt(1 - gamma_base)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(gamma_base)],
                   [0, 0]], dtype=complex)
    K_ops = [K0, K1]

    # Single-qubit amplitude damping
    for qubit in range(n_qubits):
        rho_single_noisy = np.zeros_like(rho_current, dtype=complex)
        for K in K_ops:
            oplist = [I2] * n_qubits
            oplist[qubit] = K
            K_full = build_full_operator_from_list(oplist)
            rho_single_noisy += K_full @ rho_current @ K_full.conj().T
        rho_current = rho_single_noisy

    # Correlated two-qubit amplitude damping
    for qubit1 in range(n_qubits - 1):
        qubit2 = qubit1 + 1
        distance = 1.0
        p_corr = correlation_strength * gamma_base * np.exp(-distance / correlation_length)
        if p_corr < 1e-12:
            continue

        rho_2q_noisy = (1.0 - p_corr) * rho_current
        for K1a, K1b in [(K0, K0), (K0, K1), (K1, K0), (K1, K1)]:
            oplist = [I2] * n_qubits
            oplist[qubit1] = K1a
            oplist[qubit2] = K1b
            K_full = build_full_operator_from_list(oplist)
            rho_2q_noisy += (p_corr / 4.0) * (K_full @ rho_current @ K_full.conj().T)
        rho_current = rho_2q_noisy

    tr = np.trace(rho_current)
    if abs(tr) > 1e-12:
        rho_current = rho_current / tr
    return rho_current


def apply_correlated_phase_damping(rho_ideal,
                                   n_qubits,
                                   lambda_base,
                                   correlation_length,
                                   correlation_strength):
    rho_current = rho_ideal.copy()

    K0 = np.array([[1, 0],
                   [0, np.sqrt(1 - lambda_base)]], dtype=complex)
    K1 = np.array([[0, 0],
                   [0, np.sqrt(lambda_base)]], dtype=complex)
    K_ops = [K0, K1]

    # Single-qubit phase damping
    for qubit in range(n_qubits):
        rho_single_noisy = np.zeros_like(rho_current, dtype=complex)
        for K in K_ops:
            oplist = [I2] * n_qubits
            oplist[qubit] = K
            K_full = build_full_operator_from_list(oplist)
            rho_single_noisy += K_full @ rho_current @ K_full.conj().T
        rho_current = rho_single_noisy

    # Correlated two-qubit phase damping
    for qubit1 in range(n_qubits - 1):
        qubit2 = qubit1 + 1
        distance = 1.0
        p_corr = correlation_strength * lambda_base * np.exp(-distance / correlation_length)
        if p_corr < 1e-12:
            continue

        rho_2q_noisy = (1.0 - p_corr) * rho_current
        for K1a, K1b in [(K0, K0), (K0, K1), (K1, K0), (K1, K1)]:
            oplist = [I2] * n_qubits
            oplist[qubit1] = K1a
            oplist[qubit2] = K1b
            K_full = build_full_operator_from_list(oplist)
            rho_2q_noisy += (p_corr / 4.0) * (K_full @ rho_current @ K_full.conj().T)
        rho_current = rho_2q_noisy

    tr = np.trace(rho_current)
    if abs(tr) > 1e-12:
        rho_current = rho_current / tr
    return rho_current


# TOPAZ-like hyperparams
DEPOLBASE = 0.005
DEPOLCORRLENGTH = 2.0
DEPOLCORRSTRENGTH = 0.3

AMPDAMPBASE = 0.008
AMPDAMPCORRLENGTH = 2.0
AMPDAMPCORRSTRENGTH = 0.25

PHASEDAMPBASE = 0.012
PHASEDAMPCORRLENGTH = 2.0
PHASEDAMPCORRSTRENGTH = 0.3


def apply_all_correlated_noises(rho_ideal, n_qubits):
    rho = rho_ideal.copy()
    rho = apply_correlated_depolarizing_noise(
        rho, n_qubits,
        DEPOLBASE, DEPOLCORRLENGTH, DEPOLCORRSTRENGTH
    )
    rho = apply_correlated_amplitude_damping(
        rho, n_qubits,
        AMPDAMPBASE, AMPDAMPCORRLENGTH, AMPDAMPCORRSTRENGTH
    )
    rho = apply_correlated_phase_damping(
        rho, n_qubits,
        PHASEDAMPBASE, PHASEDAMPCORRLENGTH, PHASEDAMPCORRSTRENGTH
    )
    return rho


# =====================================================
#  CUSTOM ESTIMATOR WRAPPER — FIXED
# =====================================================
class NoisyStatevectorEstimator:
    """
    Estimator-style interface that:
      - binds params to ansatz circuit using a param->value dict,
      - simulates statevector,
      - applies correlated noise to the density matrix,
      - returns <H> = Tr(ρ H).

    Fixes vs original:
      1. params may arrive as a 2-D array shape (1, n_params) — flatten first.
      2. Use assign_parameters with a {Parameter: value} dict, which is the
         only reliable path for UCCSD / ParameterVector circuits.
      3. Observable may be SparsePauliOp or already a matrix; handle both.
    """

    def __init__(self):
        pass

    @staticmethod
    def _bind(circ, params):
        """Return a fully-bound circuit regardless of how params arrive."""
        # Flatten to a 1-D float array
        param_values = np.asarray(params, dtype=float).flatten()

        # Get the circuit's free parameters in canonical sorted order
        sorted_params = sorted(circ.parameters, key=lambda p: p.name)

        if len(sorted_params) != len(param_values):
            raise ValueError(
                f"Parameter count mismatch: circuit has {len(sorted_params)} "
                f"free params but {len(param_values)} values were supplied."
            )

        # Build a {Parameter: float} mapping and bind
        param_dict = dict(zip(sorted_params, param_values))
        return circ.assign_parameters(param_dict, inplace=False)

    @staticmethod
    def _observable_matrix(obs):
        """Return the full Hermitian matrix for obs (SparsePauliOp or ndarray)."""
        if isinstance(obs, np.ndarray):
            return obs
        # SparsePauliOp (and similar Qiskit operator types) expose .to_matrix()
        return obs.to_matrix()

    def run(self, pubs):
        """
        pubs : iterable of (circuit, observable, param_values) tuples.
        Returns a Job whose .result()[i].data.evs holds the expectation value.
        """
        pub_results = []

        for pub in pubs:
            circ, obs, params = pub[0], pub[1], pub[2]

            # --- 1. Bind parameters ---
            bound_circ = self._bind(circ, params)

            # --- 2. Statevector simulation ---
            sv = Statevector.from_instruction(bound_circ)
            vec = sv.data
            rho_ideal = np.outer(vec, np.conjugate(vec))

            # --- 3. Apply correlated noise ---
            n_qubits = bound_circ.num_qubits
            rho_noisy = apply_all_correlated_noises(rho_ideal, n_qubits)

            # --- 4. Expectation value Tr(ρ H) ---
            H = self._observable_matrix(obs)
            ev = float(np.real(np.trace(rho_noisy @ H)))

            # --- 5. Wrap in the result structure VQE expects ---
            _ev = ev  # capture for closure

            class _Data:
                evs = np.array([_ev])

            class _PubResult:
                data = _Data()

            pub_results.append(_PubResult())

        # ---- Job / Result wrappers ----
        _pub_results = pub_results  # capture for closures

        class _Result:
            def __getitem__(self, idx):
                return _pub_results[idx]

        class _Job:
            def result(self):
                return _Result()

        return _Job()


# =====================================================
#  H₂ MOLECULE SETUP
# =====================================================
print("\n PySCF electronic structure...")
driver = PySCFDriver(atom='H 0 0 0; H 0 0 0.74', basis='sto3g')
esp = driver.run()

print(f"   Spatial orbitals: {esp.num_spatial_orbitals}")
print(f"   Particles: {esp.num_particles}")
print(f"   Nuclear repulsion: {esp.nuclear_repulsion_energy:.6f} Ha")


# =====================================================
# ELECTRONIC HAMILTONIAN → PAULI
# =====================================================
print("\n Jordan-Wigner transformation...")
mapper = JordanWignerMapper()
second_q_op = esp.hamiltonian.second_q_op()
qubit_op = mapper.map(second_q_op)
print(f"   Pauli terms: {len(qubit_op)}")
print(f"   Qubits: {qubit_op.num_qubits}")


# =====================================================
#  HARTREE-FOCK + UCCSD ANSATZ
# =====================================================
print("\n Hartree-Fock + UCCSD...")
hf = HartreeFock(esp.num_spatial_orbitals, esp.num_particles, mapper)
ansatz = UCCSD(esp.num_spatial_orbitals, esp.num_particles, mapper, initial_state=hf)
print(f"   Parameters: {ansatz.num_parameters}")


# =====================================================
# VQE OPTIMIZATION WITH YOUR NOISE
# =====================================================
print("\n SLSQP variational optimization under custom correlated noise...")
estimator = NoisyStatevectorEstimator()
optimizer = SLSQP(maxiter=200)

vqe = VQE(estimator, ansatz, optimizer)
vqe.initial_point = np.zeros(ansatz.num_parameters)

print("   Computing ⟨ψ|H_electronic|ψ⟩ with noisy ρ...")
result = vqe.compute_minimum_eigenvalue(qubit_op)


# =====================================================
#  TOTAL MOLECULAR ENERGY
# =====================================================
print("\n" + "="*70)
print(" H₂ GROUND STATE ENERGIES (with custom correlated noise)")
print("="*70)
electronic_energy = float(result.eigenvalue.real)
nuclear_repulsion = esp.nuclear_repulsion_energy
total_energy = electronic_energy + nuclear_repulsion

print(f" Electronic (VQE, noisy): {electronic_energy:.6f} Ha")
print(f"  Nuclear repulsion:      {nuclear_repulsion:.6f} Ha")
print(f" TOTAL molecular:         {total_energy:.6f} Ha")
print(f" Expected ideal total:    -1.137 Ha")
print(f" Error vs ideal:          {abs(total_energy + 1.137):.1e} Ha")

print(f"  Params optimized:       {len(result.optimal_parameters)}")
print(f"  Iterations:             {result.optimizer_result.nfev}")

 FULL H₂ VQE — Electronic + Nuclear Repulsion (with custom correlated noise)

 PySCF electronic structure...
   Spatial orbitals: 2
   Particles: (1, 1)
   Nuclear repulsion: 0.715104 Ha

 Jordan-Wigner transformation...
   Pauli terms: 15
   Qubits: 4

 Hartree-Fock + UCCSD...
   Parameters: 3

 SLSQP variational optimization under custom correlated noise...
   Computing ⟨ψ|H_electronic|ψ⟩ with noisy ρ...

 H₂ GROUND STATE ENERGIES (with custom correlated noise)
 Electronic (VQE, noisy): -1.833580 Ha
  Nuclear repulsion:      0.715104 Ha
 TOTAL molecular:         -1.118475 Ha
 Expected ideal total:    -1.137 Ha
 Error vs ideal:          1.9e-02 Ha
  Params optimized:       3
  Iterations:             10
